In [9]:
import zipfile
import shutil
from pathlib import Path
import csv
import json

from PIL import Image
import numpy as np
import pytesseract
import cv2  # opencv-python

import xml.etree.ElementTree as ET


# =========================
# CONFIG
# =========================

# Path to tesseract executable on Windows
# Adjust if installed somewhere else.
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# OCR / ADA thresholds
OCR_CONF_THRESHOLD = 60        # minimum OCR confidence for a text box
MIN_CONTRAST_PASS = 4.5        # WCAG AA small text
MIN_TEXT_HEIGHT_PX = 14        # rough threshold for readable text
BLUR_THRESHOLD = 80.0          # variance of Laplacian; lower = blurrier


# =========================
# UTILS: WCAG CONTRAST
# =========================

def srgb_to_linear(c):
    """Convert 0–1 sRGB channel to linear."""
    if c <= 0.04045:
        return c / 12.92
    return ((c + 0.055) / 1.055) ** 2.4

def relative_luminance(rgb):
    """rgb is (R,G,B) in 0–255."""
    r, g, b = [x / 255.0 for x in rgb]
    r_lin = srgb_to_linear(r)
    g_lin = srgb_to_linear(g)
    b_lin = srgb_to_linear(b)
    return 0.2126 * r_lin + 0.7152 * g_lin + 0.0722 * b_lin

def contrast_ratio(c1, c2):
    """Return WCAG contrast ratio between two RGB colors."""
    L1 = relative_luminance(c1)
    L2 = relative_luminance(c2)
    L_light = max(L1, L2)
    L_dark = min(L1, L2)
    return (L_light + 0.05) / (L_dark + 0.05)


# =========================
# STEP 1: Extract images
# =========================

def extract_images(docx_path, out_dir):
    """
    Extract embedded images from word/media/* in a DOCX.
    """
    docx_path = Path(docx_path)
    out_dir = Path(out_dir)
    media_dir = out_dir / "media"
    media_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with zipfile.ZipFile(docx_path, 'r') as z:
        for file in z.namelist():
            if file.startswith("word/media/"):
                filename = Path(file).name
                target = media_dir / filename

                with z.open(file) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                print(f"Extracted: {file} -> {target}")
                extracted.append(filename)

    return extracted, media_dir


# =========================
# OPTIONAL: Extract alt text
# =========================

def extract_alt_text_map(docx_path):
    """
    Return a dict: { image_filename: alt_text }
    using document.xml and its relationships.
    """
    docx_path = Path(docx_path)
    alt_map = {}

    with zipfile.ZipFile(docx_path, 'r') as z:
        ns = {
            "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
            "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
            "a":  "http://schemas.openxmlformats.org/drawingml/2006/main",
            "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture"
        }

        # read relationships to map rId -> image file
        rels = {}
        rels_xml = z.read("word/_rels/document.xml.rels")
        rels_root = ET.fromstring(rels_xml)
        rel_ns = {"": "http://schemas.openxmlformats.org/package/2006/relationships"}

        for rel in rels_root.findall("Relationship", rel_ns):
            if rel.attrib.get("Type", "").endswith("/image"):
                rels[rel.attrib["Id"]] = Path(rel.attrib["Target"]).name

        # parse main document xml for pic elements
        doc_xml = z.read("word/document.xml")
        root = ET.fromstring(doc_xml)

        for pic in root.findall(".//pic:pic", ns):
            cnvpr = pic.find("pic:nvPicPr/pic:cNvPr", ns)
            if cnvpr is None:
                continue

            alt_text = cnvpr.attrib.get("descr", "")
            # name = cnvpr.attrib.get("name", "")

            blip = pic.find(".//a:blip", ns)
            if blip is not None:
                embed = blip.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}embed")
                image_file = rels.get(embed)
                if image_file:
                    alt_map[image_file] = alt_text

    return alt_map


# =========================
# STEP 2: ADA analysis per image
# =========================

def analyze_image_for_ada(image_path):
    """
    Run OCR + ADA checks on a single image.
    Returns a dict of metrics.
    """
    img_path = Path(image_path)
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    h, w, _ = img_np.shape

    # OCR with detailed data
    ocr_data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

    has_text = False
    text_boxes = []
    min_contrast = None
    min_text_height = None

    # Loop over words
    n = len(ocr_data["text"])
    for i in range(n):
        text = ocr_data["text"][i].strip()
        conf = int(ocr_data["conf"][i])

        if not text:
            continue
        if conf < OCR_CONF_THRESHOLD:
            continue

        has_text = True
        x = ocr_data["left"][i]
        y = ocr_data["top"][i]
        w_box = ocr_data["width"][i]
        h_box = ocr_data["height"][i]

        text_boxes.append((text, conf, x, y, w_box, h_box))

        # track smallest text height
        if min_text_height is None or h_box < min_text_height:
            min_text_height = h_box

        # sample foreground (center of box) and background (slightly outside)
        cx = min(x + w_box // 2, w - 1)
        cy = min(y + h_box // 2, h - 1)
        fg_color = img_np[cy, cx, :]  # RGB

        # background sample: above box center (if possible)
        by = max(y - 2, 0)
        bx = cx
        bg_color = img_np[by, bx, :]

        cr = contrast_ratio(fg_color, bg_color)
        if min_contrast is None or cr < min_contrast:
            min_contrast = cr

    # Blur detection via variance of Laplacian
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    blur_metric = lap.var()

    # Build result
    result = {
        "image_file": img_path.name,
        "has_text": has_text,
        "min_contrast": float(min_contrast) if min_contrast is not None else None,
        "min_text_height_px": int(min_text_height) if min_text_height is not None else None,
        "blur_metric": float(blur_metric),
        "ocr_text_boxes": [
            {
                "text": t,
                "conf": c,
                "bbox": [int(x), int(y), int(wb), int(hb)]
            }
            for (t, c, x, y, wb, hb) in text_boxes
        ]
    }

    # Simple pass/fail flags
    contrast_ok = (min_contrast is None) or (min_contrast >= MIN_CONTRAST_PASS)
    text_size_ok = (min_text_height is None) or (min_text_height >= MIN_TEXT_HEIGHT_PX)
    blur_ok = blur_metric >= BLUR_THRESHOLD

    result["contrast_ok"] = contrast_ok
    result["text_size_ok"] = text_size_ok
    result["blur_ok"] = blur_ok
    result["ada_pass"] = contrast_ok and text_size_ok and blur_ok

    return result


# =========================
# STEP 3: Run on one DOCX
# =========================

def run_pipeline_for_docx(docx_file, output_root):
    docx_file = Path(docx_file)
    output_root = Path(output_root)
    doc_out_dir = output_root / docx_file.stem
    doc_out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Extract images
    extracted, media_dir = extract_images(docx_file, doc_out_dir)

    # 2) Optional alt text map
    alt_map = extract_alt_text_map(docx_file)

    # 3) Run ADA analysis on each extracted image
    results = []
    for img_name in extracted:
        img_path = media_dir / img_name
        ada = analyze_image_for_ada(img_path)
        ada["alt_text"] = alt_map.get(img_name, "")
        results.append(ada)

    # 4) Write CSV, JSON, HTML
    write_reports(doc_out_dir, results)

    print("\nPipeline complete for:", docx_file)
    print("Output folder:", doc_out_dir)


# =========================
# STEP 4: Reporting
# =========================

def write_reports(out_dir, results):
    out_dir = Path(out_dir)
    # CSV
    csv_path = out_dir / "ada_report.csv"
    fieldnames = [
        "image_file",
        "has_text",
        "min_contrast",
        "min_text_height_px",
        "blur_metric",
        "contrast_ok",
        "text_size_ok",
        "blur_ok",
        "ada_pass",
        "alt_text",
    ]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in results:
            row = {k: r.get(k) for k in fieldnames}
            w.writerow(row)

    # JSON (full detail including OCR boxes)
    json_path = out_dir / "ada_report.json"
    with open(json_path, "w", encoding="utf-8") as f:
      #  json.dump(results, f, indent=2)

                # safe_results = []
                # for r in results:
                #     safe_r = {k: make_json_safe(v) for k, v in r.items()}
                #     # Also fix OCR bounding boxes inside
                #     if "ocr_text_boxes" in r:
                #         safe_r["ocr_text_boxes"] = [
                #             {sk: make_json_safe(sv) for sk, sv in box.items()}
                #             for box in r["ocr_text_boxes"]
                #         ]
                #     safe_results.append(safe_r)
        safe_results = make_json_safe(results)
        json.dump(safe_results, f, indent=2)


    # HTML
    html_path = out_dir / "ada_report.html"
    print("Writing HTML report to ", html_path)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'><title>ADA Image Report</title></head><body>\n")
        f.write("<h1>ADA Image Report</h1>\n")
        f.write("<table border='1' cellpadding='4' cellspacing='0'>\n")
        f.write("<tr><th>Image</th><th>Has Text</th><th>Min Contrast</th>"
                "<th>Min Text Height (px)</th><th>Blur Metric</th>"
                "<th>Contrast OK</th><th>Text Size OK</th><th>Blur OK</th>"
                "<th>ADA PASS</th><th>Alt Text</th></tr>\n")

        for r in results:
            img_rel = f"media/{r['image_file']}"
            f.write("<tr>")
            f.write(f"<td><img src='{img_rel}' style='max-width:300px; max-height:200px;'><br>{r['image_file']}</td>")
            f.write(f"<td>{r['has_text']}</td>")
            f.write(f"<td>{r['min_contrast']}</td>")
            f.write(f"<td>{r['min_text_height_px']}</td>")
            f.write(f"<td>{r['blur_metric']:.1f}</td>")
            f.write(f"<td>{r['contrast_ok']}</td>")
            f.write(f"<td>{r['text_size_ok']}</td>")
            f.write(f"<td>{r['blur_ok']}</td>")
            f.write(f"<td style='font-weight:bold; color:{'green' if r['ada_pass'] else 'red'}'>{r['ada_pass']}</td>")
            f.write(f"<td>{(r.get('alt_text') or '').replace('<','&lt;').replace('>','&gt;')}</td>")
            f.write("</tr>\n")

        f.write("</table>\n</body></html>\n")

    print("Wrote:", csv_path)
    print("Wrote:", json_path)
    print("Wrote:", html_path)

def make_json_safe(value):
    # Handle numpy scalar types (bool_, int_, float_, etc.)
    if isinstance(value, np.generic):
        return value.item()   # converts to a native Python type

    # Handle lists/dicts recursively
    if isinstance(value, list):
        return [make_json_safe(v) for v in value]
    if isinstance(value, dict):
        return {k: make_json_safe(v) for k, v in value.items()}

    # Already a native Python type
    return value
# =========================
# MAIN: EDIT THESE PATHS
# =========================

if __name__ == "__main__":
    DOCX_FILE =  r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25.docx"

    OUTPUT_ROOT = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports"

    run_pipeline_for_docx(DOCX_FILE, OUTPUT_ROOT)


Extracted: word/media/image1.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image1.png
Extracted: word/media/image2.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image2.png
Extracted: word/media/image3.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image3.png
Extracted: word/media/image4.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image4.png
Extracted: word/media/image5.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image5.png
Extracted: word/media/image6.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Repor